**Silver Data Ingestion**

In [0]:
import pyspark.sql.functions as F
from typing import List
from pyspark.sql import DataFrame, Window
from delta.tables import DeltaTable

In [0]:
class Transformations:
    @staticmethod
    def dedup(df: DataFrame, dedup_cols: List[str], cdc: str) -> DataFrame:
        w = Window.partitionBy(*dedup_cols).orderBy(
            F.col(cdc).desc(), F.col("_ingested_at").desc())
        return (df.withColumn("_rn", F.row_number().over(w))
                  .filter("_rn = 1")
                  .drop("_rn"))
    @staticmethod    
    def make_upsert(entity, pk, cdc):
        target = f"puspark_dbt.silver.{entity}"

        def upsert(batch_df, batch_id):
            latest = Transformations.dedup(batch_df, pk, cdc)

            if not spark.catalog.tableExists(target):
                latest.write.format("delta").saveAsTable(target)
                return

            cond = " AND ".join(f"t.{k} = s.{k}" for k in pk)

            (DeltaTable.forName(spark, target).alias("t")
                .merge(latest.alias("s"), cond)
                .whenMatchedUpdateAll(condition=f"s.{cdc} >= t.{cdc}")
                .whenNotMatchedInsertAll()
                .execute())
        return upsert

In [0]:
df_cust= spark.read.table('puspark_dbt.bronze.customers')

df_cust= df_cust.withColumn('phone_number', F.regexp_replace('phone_number', r'[^0-9]', ''))

df_cust= df_cust.withColumn('Full_name' , F.concat_ws(' ',F.col('first_name') , F.col('last_name')))

df_cust= df_cust.drop('first_name','last_name')



In [0]:


config = {
    "customers": {"pk": ["customer_id"], "cdc": "last_updated_timestamp"},
    "drivers":   {"pk": ["driver_id"],   "cdc": "_ingested_at"},
    "vehicles":  {"pk": ["vehicle_id"],  "cdc": "_ingested_at"},
    "trips":     {"pk": ["trip_id"],     "cdc": "_ingested_at"},
    "payments":  {"pk": ["payment_id"],  "cdc": "_ingested_at"},
    "locations": {"pk": ["location_id"], "cdc": "_ingested_at"},
}
base = "/Volumes/puspark_dbt"
for entity , cfg in config.items():
    q=(spark.readStream.table(f'puspark_dbt.bronze.{entity}')
        .writeStream.foreachBatch(Transformations.make_upsert(entity,cfg['pk'] , cfg['cdc']))
        .option('checkpointLocation', f"{base}/silver/checkpoint/{entity}")
        .trigger(availableNow=True)
        .start())
    
    q.awaitTermination()


###Incremntal and upsert testing on customers table###

In [0]:
%sql
   select count(*) from puspark_dbt.silver.customers;   

count(*)
260


In [0]:
%sql

select customer_id, city, email, phone_number, last_updated_timestamp
from puspark_dbt.silver.customers
where customer_id in (1, 2, 3, 4, 5, 6, 1001)
order by customer_id;

customer_id,city,email,phone_number,last_updated_timestamp
1,Westhaven,azimmerman@ramirez-nelson.com,811-724-1080,2025-10-02T09:00:00.000Z
2,Baldwinburgh,williammiller@serrano-jones.com,632-190-4027,2025-10-02T09:05:00.000Z
3,New Ashley,troy07@carrillo-webb.info,676.094.0716,2025-09-13T03:45:53.000Z
4,Stephanieton,victor.sanchez@newmail.com,960-929-2694,2025-10-02T09:10:00.000Z
5,Riverton,patricknewman@williams.biz,621-699-9458,2025-10-02T09:15:00.000Z
6,East Pamela,courtney29@gmail.com,332-469-2812,2025-10-02T09:20:00.000Z
1001,Hillcrest,omar.khan1001@example.com,555-696-1950,2025-10-03T10:00:00.000Z
